<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Synthetic Financial Time Series

**GAN-based Generation of Realistic Return Data (PyTorch)**

&copy; Dr. Yves J. Hilpisch

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a>

Install the package before running this notebook. Set `SYNTHTS_DATA` to
a frozen local CSV to avoid a live download. Sequential/TimeGAN examples
accept an explicitly selected YAML via `SYNTHTS_CONFIG`; historical
selected settings and their recorded metrics are not loaded automatically.

## Imports & Settings

In [ ]:
import os
import numpy as np
import pandas as pd
from pylab import plt, mpl

In [ ]:
import torch
from torch import nn

In [ ]:
from synthts.data import (
    EOD_URL, load_eod, compute_log_returns, ReturnScaler, split_returns,
)
from synthts.vanilla import (
    Discriminator, GAN, Generator, generate_samples, train_gan,
)
from synthts.evaluate import (ks_test, ks_test_batch, pass_rate,
    describe_comparison, mse_sorted,
    plot_real_vs_synth, plot_histograms, plot_sorted,
    plot_cumulative, plot_pvalue_histogram)

In [ ]:
plt.style.use('seaborn-v0_8')
mpl.rcParams['figure.dpi'] = 300
mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['font.family'] = 'serif'

## Simple Example — Learning a Cubic Function

In [ ]:
x = np.linspace(-2, 2, 500)
y = x ** 3

In [ ]:
scaler = ReturnScaler()
y_ = scaler.fit_transform(y)

In [ ]:
plt.plot(x, y, 'r', lw=1.0, label='real data')
plt.plot(x, y_, 'b--', lw=1.0, label='normalised data')
plt.legend();

### Model Training

In [ ]:
torch.manual_seed(100)
generator = Generator(latent_dim=1, hidden_units=32, output_dim=1)
discriminator = Discriminator(input_dim=1, hidden_units=32)

In [ ]:
%%time
history = train_gan(
    generator, discriminator, y_.reshape(-1, 1),
    epochs=15001, batch_size=32,
    lr_d=0.001, lr_g=0.0001, seed=100
)

### Evaluation

In [ ]:
data = pd.DataFrame({'real': y}, index=x)
N = 5
for i in range(N):
    samples = generate_samples(generator, len(y), latent_dim=1)
    data[f'synth_{i:02d}'] = scaler.inverse_transform(samples)

In [ ]:
data.describe().round(3)

In [ ]:
mse_sorted(data['real'].values, data['synth_00'].values)

In [ ]:
data.apply(np.sort).plot(
    style=['r'] + N * ['b--'], lw=1, legend=False);

## Financial Example — Real Log-Returns

In [ ]:
SYM = 'SPY'

In [ ]:
prices = load_eod(url=os.environ.get('SYNTHTS_DATA', EOD_URL),
                  symbol=SYM, n_obs=2 * 252)
rets = compute_log_returns(prices)

In [ ]:
scaler = ReturnScaler()
rets_ = scaler.fit_transform(rets)

In [ ]:
torch.manual_seed(100)
generator = Generator(latent_dim=1, hidden_units=16, output_dim=1)
discriminator = Discriminator(input_dim=1, hidden_units=16)

In [ ]:
%%time
history = train_gan(
    generator, discriminator, rets_.reshape(-1, 1),
    epochs=15001, batch_size=32,
    lr_d=0.001, lr_g=0.0001, seed=100
)

### Generating Synthetic Returns

In [ ]:
data = pd.DataFrame({'real': rets})
N = 25
for i in range(N):
    samples = generate_samples(generator, len(rets_), latent_dim=1)
    data[f'synth_{i:02d}'] = scaler.inverse_transform(samples)

In [ ]:
describe_comparison(data, n_show=5)

In [ ]:
plot_real_vs_synth(data, n_synth=1);

In [ ]:
plot_histograms(data['real'].values, data['synth_00'].values);

In [ ]:
plot_sorted(data['real'].values, data['synth_00'].values);

In [ ]:
plot_cumulative(data, n_synth=N);

### Kolmogorov-Smirnov Test

Non-rejection is a descriptive diagnostic, not evidence of equivalence.
Serial dependence and overlapping windows prevent interpreting these
p-values as calibrated independent-sample tests.

In [ ]:
synth_cols = [c for c in data.columns if c.startswith('synth')]
pvals = ks_test_batch(data['real'].values, data[synth_cols])
pvals.round(4)

In [ ]:
f'KS pass rate: {pass_rate(pvals):.0%}'

In [ ]:
plot_pvalue_histogram(pvals);

<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a>